# Análisis exploratorio del Censo de Locales y Actividades de Madrid

**TFM — Predicción de rotación comercial.** Fase 1 de la guía: *análisis descriptivo
del conjunto, gráfico en lo posible*.

Este cuaderno mira los datos **antes** de modelar. No entrena nada y no toca el
campeón congelado. Trabaja sobre dos insumos ya construidos por el pipeline:

| insumo | qué es | lo produce |
|---|---|---|
| `datos/panel/panel_AAAA.parquet` | 13 cortes anuales del censo (2014–2026), un local por fila | `src/hito3b_cohortes.py` |
| `datos/dataset_modelado.pkl` | target + 21 variables por cohorte (train/val/test/covid) | `src/hito4_variables.py` |

Se ejecuta **desde la raíz del repositorio**. Las cuatro figuras que se guardan
en `salida/figuras/eda_*.png` son las que van a la memoria; el resto del
material vive solo en este HTML.

> Guía, advertencia recogida: *no incluir listados largos de datos sin valor*.
> Aquí todo listado está agregado o recortado a lo relevante.

In [1]:
import warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 120, "font.size": 10,
                     "axes.grid": True, "grid.alpha": .3, "figure.autolayout": True})

RAIZ = Path.cwd()
assert (RAIZ / "datos" / "panel").exists(), f"ejecuta el notebook desde la raiz del repo (cwd={RAIZ})"
FIG = RAIZ / "salida" / "figuras"; FIG.mkdir(parents=True, exist_ok=True)

ANIOS = list(range(2014, 2027))

def a_numero(serie):
    return pd.to_numeric(serie.astype(str).str.replace(",", ".", regex=False).str.strip(),
                         errors="coerce")

def cargar_panel(anio, columnas=None):
    return pd.read_parquet(RAIZ / "datos" / "panel" / f"panel_{anio}.parquet", columns=columnas)

FICHA = json.loads((RAIZ / "datos" / "ficha_modelo.json").read_text(encoding="utf-8"))
print("campeon congelado:", FICHA["modelo"])
print("variables del campeon:", FICHA["variables"])

campeon congelado: C. Sector + distrito + historia
variables del campeon: ['id_epigrafe', 'id_division', 'desc_distrito_local', 'desc_tipo_acceso_local', 'antiguedad_negocio', 'antiguedad_local', 'rotaciones_previas', 'tasa_rotacion_local', 'antiguedad_censurada']


## 1. Volumen del censo por año y su evolución

Cuántos registros trae cada corte anual y cuántos corresponden a un local
**abierto**. El censo no es una foto de "locales activos": incluye cerrados,
locales en obras y bajas. La serie de *abiertos* es la base sobre la que se
mide la rotación.

In [2]:
filas = []
for a in ANIOS:
    p = cargar_panel(a, ["id_local", "abierto", "desc_situacion_local"])
    filas.append({"anio": a, "registros": len(p), "abiertos": int(p["abierto"].sum())})
vol = pd.DataFrame(filas).set_index("anio")
vol["% abiertos"] = (vol["abiertos"] / vol["registros"] * 100).round(1)
vol["var. abiertos %"] = (vol["abiertos"].pct_change() * 100).round(2)
display(vol)

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(vol.index, vol["registros"], "o-", label="registros del censo")
ax.plot(vol.index, vol["abiertos"], "s-", label="locales abiertos")
ax.set_title("Volumen del Censo de Locales de Madrid, 2014-2026")
ax.set_xlabel(""); ax.set_ylabel("nº de locales")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1000:.0f}k"))
ax.legend(); ax.set_xticks(ANIOS); ax.tick_params(axis="x", rotation=45)
fig.savefig(FIG / "eda_volumen_censo.png", bbox_inches="tight"); plt.close(fig)
print("guardada salida/figuras/eda_volumen_censo.png")

,registros,abiertos,% abiertos,var. abiertos %
anio,,,,
2014,142484,94579,66.4,NaN
2015,144168,97674,67.8,3.27
2016,145065,99185,68.4,1.55
2017,145752,100227,68.8,1.05
2018,146716,101525,69.2,1.30
2019,147576,102434,69.4,0.90
2020,148033,102403,69.2,-0.03
2021,149935,99083,66.1,-3.24
2022,150522,99488,66.1,0.41


guardada salida/figuras/eda_volumen_censo.png


**Lectura.** El censo crece de forma sostenida (~142k → ~152k registros). Los
locales abiertos pasan de ~95k a ~100k, con un escalón visible entre 2020 y
2021 (efecto de la pandemia sobre los cierres, no una caída de cobertura). El
salto de registros de 2021→2022 coincide con la incorporación de la sección
censal al fichero (ver §5).

## 2. Distribución de locales por distrito, barrio y sección de actividad

Foto del **panel 2021** (la cohorte de test), locales abiertos. Interesa el
grado de concentración: si unos pocos distritos / secciones acumulan la mayor
parte del comercio, el modelo tiene que aprender sobre todo de ellos.

In [3]:
p21 = cargar_panel(2021)
ab21 = p21[p21["abierto"]].copy()
ab21["distrito"] = ab21["desc_distrito_local"].astype(str).str.strip().str.title()
ab21["barrio"]   = ab21["desc_barrio_local"].astype(str).str.strip().str.title()
ab21["seccion"]  = ab21["desc_seccion"].astype(str).str.strip()
print(f"locales abiertos en 2021: {len(ab21):,}")

por_distrito = ab21["distrito"].value_counts()
por_barrio   = ab21["barrio"].value_counts()
por_seccion  = ab21["seccion"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
por_distrito.sort_values().plot.barh(ax=axes[0], color="#4C72B0")
axes[0].set_title(f"Locales abiertos por distrito (2021) · {len(por_distrito)} distritos")
axes[0].set_xlabel("nº de locales")
top_sec = por_seccion.head(10).sort_values()
top_sec.index = [s if len(s) < 45 else s[:42] + "..." for s in top_sec.index]
top_sec.plot.barh(ax=axes[1], color="#55A868")
axes[1].set_title("10 secciones de actividad más frecuentes (2021)")
axes[1].set_xlabel("nº de locales")
plt.show(); plt.close(fig)

print("\nConcentración por distrito:")
cd = (por_distrito.cumsum() / por_distrito.sum() * 100).round(1)
print(f"  los 5 distritos con más comercio ({', '.join(por_distrito.head(5).index)})")
print(f"  acumulan el {cd.iloc[4]:.0f}% de los locales abiertos.")
print("\nConcentración por sección de actividad:")
print(f"  Comercio + Hostelería solas = {por_seccion.iloc[:2].sum()/por_seccion.sum()*100:.0f}% del total.")
print(f"\nBarrios: {ab21['barrio'].nunique()} barrios; mediana de {por_barrio.median():.0f} "
      f"locales/barrio, del más pequeño ({por_barrio.min()}) al mayor ({por_barrio.max()}).")

locales abiertos en 2021: 99,083

Concentración por distrito:
  los 5 distritos con más comercio (Centro, Carabanchel, Salamanca, Ciudad Lineal, Chamberi)
  acumulan el 38% de los locales abiertos.

Concentración por sección de actividad:
  Comercio + Hostelería solas = 56% del total.

Barrios: 131 barrios; mediana de 683 locales/barrio, del más pequeño (17) al mayor (2579).


**Lectura.** El comercio se reparte de forma desigual pero sin un dominio
extremo: los 5 distritos con más locales reúnen ~38%. En cambio **dos
secciones de actividad** (comercio minorista y hostelería) suman más de la
mitad de los locales. Esa concentración sectorial es la que justifica que el
modelo trate el sector con detalle (epígrafe, no solo división).

## 3. Los 20 epígrafes más frecuentes y su peso

El epígrafe es la variable de actividad de mayor resolución (~440 valores). Aquí
se ve cuánta masa acumulan los más comunes y cuántos epígrafes son "cola larga"
— los que luego el codificador del modelo agrupa en un cajón `resto`
(`src/hito6_boosting.py`, `salida/DATOS.md §6`).

In [4]:
epi = (ab21.assign(epi=ab21["desc_epigrafe"].astype(str).str.strip())
       .groupby("epi").size().sort_values(ascending=False))
top20 = epi.head(20)
peso = (top20.sum() / epi.sum() * 100)

fig, ax = plt.subplots(figsize=(9, 7))
top20.sort_values().plot.barh(ax=ax, color="#C44E52")
ax.set_title(f"20 epígrafes más frecuentes (2021) — {peso:.0f}% de los locales abiertos")
ax.set_xlabel("nº de locales")
ax.set_yticklabels([s.get_text() if len(s.get_text()) < 40 else s.get_text()[:37] + "..."
                    for s in ax.get_yticklabels()])
plt.show(); plt.close(fig)

n_1pct = int((epi / epi.sum() * 100 >= 1).sum())
cola = int((epi <= 20).sum())
print(f"epígrafes distintos en 2021: {len(epi)}")
print(f"  solo {n_1pct} superan el 1% de los locales cada uno")
print(f"  {cola} epígrafes tienen 20 locales o menos (cola larga -> cajón 'resto' del modelo)")

epígrafes distintos en 2021: 436
  solo 22 superan el 1% de los locales cada uno
  130 epígrafes tienen 20 locales o menos (cola larga -> cajón 'resto' del modelo)


## 4. Situación de los locales (abierto / cerrado / baja) y su evolución

`desc_situacion_local` tiene muchas etiquetas ("Baja Reunificación", "En obras",
"Uso vivienda"…). Se agrupan en cuatro estados legibles y se mira su evolución.
El **% de cerrados** es el termómetro macro del comercio de la ciudad.

In [5]:
def agrupar_situacion(s):
    s = str(s).lower()
    if "abiert" in s: return "Abierto"
    if "cerrad" in s: return "Cerrado"
    if "vivienda" in s or "obra" in s: return "Vivienda / obras"
    return "Baja / otros"

filas = []
for a in ANIOS:
    p = cargar_panel(a, ["desc_situacion_local"])
    g = p["desc_situacion_local"].map(agrupar_situacion).value_counts(normalize=True) * 100
    filas.append({"anio": a, **g.to_dict()})
sit = pd.DataFrame(filas).set_index("anio").fillna(0)
sit = sit[["Abierto", "Cerrado", "Vivienda / obras", "Baja / otros"]].round(1)
display(sit)

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.stackplot(sit.index, [sit[c] for c in sit.columns], labels=sit.columns,
             colors=["#55A868", "#C44E52", "#8172B2", "#CCB974"], alpha=.85)
ax.set_title("Situación de los locales del censo, 2014-2026 (% de registros)")
ax.set_ylabel("% de registros"); ax.set_ylim(0, 100)
ax.set_xticks(ANIOS); ax.tick_params(axis="x", rotation=45)
ax.legend(loc="lower center", ncol=4, fontsize=8)
fig.savefig(FIG / "eda_situacion_evolucion.png", bbox_inches="tight"); plt.close(fig)
print("guardada salida/figuras/eda_situacion_evolucion.png")
print(f"\n% cerrados: {sit['Cerrado'].iloc[0]:.1f}% (2014) -> {sit['Cerrado'].loc[2020]:.1f}% (2020) "
      f"-> {sit['Cerrado'].iloc[-1]:.1f}% (2026)")

,Abierto,Cerrado,Vivienda / obras,Baja / otros
anio,,,,
2014,66.4,27.4,5.4,0.8
2015,67.8,25.8,5.3,1.2
2016,68.4,24.8,5.2,1.6
2017,68.8,24.2,5.1,1.9
2018,69.2,23.6,5.0,2.1
2019,69.4,23.3,4.9,2.3
2020,69.2,22.6,4.8,3.4
2021,66.1,24.7,5.5,3.7
2022,66.1,23.9,5.5,4.5


guardada salida/figuras/eda_situacion_evolucion.png

% cerrados: 27.4% (2014) -> 22.6% (2020) -> 24.3% (2026)


**Lectura.** El reparto de estados es muy estable. El % de cerrados baja
lentamente hasta 2020 y repunta a partir de la pandemia. Esta estabilidad
estructural es lo que hace viable entrenar con cohortes de 2015–2018 y seguir
prediciendo bien en 2025→2026 (`salida/RESULTADOS.md`, R2).

## 5. Cobertura de coordenadas y de rótulo por año

Dos campos son críticos y no siempre están:

- **coordenadas** `coordenada_x/y_local` — necesarias para las variables de
  vecindario. Se consideran válidas si son numéricas y > 1.000 (hay ceros y
  vacíos que son "sin dato").
- **rótulo** — el nombre comercial. Sin rótulo no se puede saber si el negocio
  cambió. En 2014 falta en ~1/3 de los locales (por eso la cadena arranca en
  2015). Antes de 2022 además hay un artefacto de OCR que sustituye letras por
  dígitos (`EDUCACI0N`, `REPARACI0N`), medido en `src/hito14_diagnostico_texto.py`.

In [6]:
filas = []
for a in ANIOS:
    p = cargar_panel(a, ["coordenada_x_local", "coordenada_y_local", "rotulo", "norm"])
    x, y = a_numero(p["coordenada_x_local"]), a_numero(p["coordenada_y_local"])
    coord_ok = (x.notna() & y.notna() & (x > 1000) & (y > 1000)).mean() * 100
    rot = p["rotulo"].astype(str).str.strip()
    rot_ok = (rot.notna() & ~rot.str.lower().isin(["", "nan", "none"])).mean() * 100
    ocr = p["norm"].astype(str).str.contains(r"[A-Z]0[A-Z]|0N\b", regex=True, na=False).mean() * 100
    filas.append({"anio": a, "% con coordenada": round(coord_ok, 1),
                  "% con rótulo": round(rot_ok, 1), "% rótulo con OCR 0/O": round(ocr, 1)})
cob = pd.DataFrame(filas).set_index("anio")
display(cob)

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(cob.index, cob["% con coordenada"], "o-", label="% con coordenada válida")
ax.plot(cob.index, cob["% con rótulo"], "s-", label="% con rótulo")
ax.axvline(2015, color="grey", ls=":", lw=1); ax.axvline(2022, color="grey", ls=":", lw=1)
ax.text(2015.1, 40, "arranque\nde la cadena", fontsize=8)
ax.text(2022.1, 30, "sección censal\n+ fin OCR", fontsize=8)
ax.set_title("Cobertura de campos clave por año"); ax.set_ylabel("% de registros")
ax.set_ylim(0, 105); ax.set_xticks(ANIOS); ax.tick_params(axis="x", rotation=45); ax.legend()
plt.show(); plt.close(fig)

,% con coordenada,% con rótulo,% rótulo con OCR 0/O
anio,,,
2014,90.5,67.6,0.0
2015,95.4,100.0,0.0
2016,95.0,100.0,0.0
2017,94.6,100.0,0.0
2018,94.2,100.0,0.0
2019,93.9,100.0,0.0
2020,93.5,100.0,0.0
2021,93.1,100.0,0.0
2022,92.8,100.0,0.0


## 6. Distribución espacial: mapa de densidad comercial

Densidad de locales abiertos en 2021 sobre coordenadas UTM (EPSG:25830). No se
usa `geopandas`: un `hexbin` sobre las coordenadas ya muestra la estructura
radial del comercio madrileño (almendra central densa, ejes y núcleos de
distrito).

In [7]:
x = a_numero(p21["coordenada_x_local"]); y = a_numero(p21["coordenada_y_local"])
m = p21["abierto"].values & x.notna().values & y.notna().values & (x > 1000).values & (y > 1000).values
xs, ys = x[m].values, y[m].values

fig, ax = plt.subplots(figsize=(8, 8))
hb = ax.hexbin(xs, ys, gridsize=70, cmap="inferno", mincnt=1, bins="log")
ax.set_title(f"Densidad de locales abiertos · Madrid 2021 ({m.sum():,} locales)")
ax.set_xlabel("X UTM (m)"); ax.set_ylabel("Y UTM (m)"); ax.set_aspect("equal")
cb = fig.colorbar(hb, ax=ax, shrink=.7); cb.set_label("locales por celda (escala log)")
fig.savefig(FIG / "eda_densidad_espacial.png", bbox_inches="tight"); plt.close(fig)
print("guardada salida/figuras/eda_densidad_espacial.png")
print(f"locales sin coordenada usable en 2021: {(~m & p21['abierto'].values).sum():,} "
      f"({(~m & p21['abierto'].values).sum()/p21['abierto'].sum()*100:.1f}% de los abiertos)")

guardada salida/figuras/eda_densidad_espacial.png
locales sin coordenada usable en 2021: 6,409 (6.5% de los abiertos)


## 7. Tasa de rotación por sector, por distrito y su cruce

Se usa `dataset_modelado.pkl`, que ya tiene el **target** (¿el local dejó de
albergar el mismo negocio al año siguiente?) por cohorte. Se excluye la cohorte
**COVID** (2020→2021), un shock atípico que no representa el régimen normal.

In [8]:
ds = pd.read_pickle(RAIZ / "datos" / "dataset_modelado.pkl")
d = ds[ds["uso"] != "covid"].copy()
base = d["target"].mean()
print(f"cohortes usadas: {sorted(d['cohorte'].unique())}   (train+val+test, sin covid)")
print(f"tasa de rotación global: {base:.2%}   ({d['target'].sum():,} de {len(d):,})")

d["division"] = d["desc_division"].astype(str).str.strip().str.title()
d["distrito"] = d["desc_distrito_local"].astype(str).str.strip().str.title()

por_div = d.groupby("division")["target"].agg(["size", "mean"])
por_div = por_div[por_div["size"] >= 2000].sort_values("mean")
por_dist = d.groupby("distrito")["target"].agg(["size", "mean"]).sort_values("mean")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
(por_div["mean"] * 100).plot.barh(ax=axes[0], color="#4C72B0")
axes[0].axvline(base * 100, color="k", ls="--", lw=1, label=f"media {base*100:.1f}%")
axes[0].set_title("Rotación por división de actividad (≥2.000 locales)")
axes[0].set_xlabel("% de rotación anual"); axes[0].legend()
axes[0].set_yticklabels([t.get_text()[:38] for t in axes[0].get_yticklabels()])
(por_dist["mean"] * 100).plot.barh(ax=axes[1], color="#55A868")
axes[1].axvline(base * 100, color="k", ls="--", lw=1)
axes[1].set_title("Rotación por distrito"); axes[1].set_xlabel("% de rotación anual")
plt.show(); plt.close(fig)

cohortes usadas: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021)]   (train+val+test, sin covid)
tasa de rotación global: 4.18%   (20,014 de 479,031)


In [9]:
# cruce sector x distrito: divisiones mas frecuentes x distritos, tasa de rotacion
top_div = d["division"].value_counts().head(8).index
sub = d[d["division"].isin(top_div)]
piv = sub.pivot_table(index="division", columns="distrito", values="target", aggfunc="mean") * 100

fig, ax = plt.subplots(figsize=(13, 4.6))
im = ax.imshow(piv.values, cmap="YlOrRd", aspect="auto", vmin=0, vmax=np.nanpercentile(piv.values, 95))
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(piv.index))); ax.set_yticklabels([s[:32] for s in piv.index], fontsize=8)
ax.set_title("Tasa de rotación anual (%) — división de actividad × distrito")
fig.colorbar(im, ax=ax, shrink=.8, label="% rotación")
fig.savefig(FIG / "eda_rotacion_sector_distrito.png", bbox_inches="tight"); plt.close(fig)
print("guardada salida/figuras/eda_rotacion_sector_distrito.png")

rango_div = (por_div["mean"].max() - por_div["mean"].min()) * 100
rango_dist = (por_dist["mean"].max() - por_dist["mean"].min()) * 100
print(f"\nrango de rotación entre divisiones: {rango_div:.1f} puntos porcentuales")
print(f"rango de rotación entre distritos:  {rango_dist:.1f} puntos porcentuales")
print("El sector separa más que el distrito: coincide con hito7 (A. Solo sector "
      "ya da lift 1,78; añadir distrito sube poco).")

guardada salida/figuras/eda_rotacion_sector_distrito.png

rango de rotación entre divisiones: 12.2 puntos porcentuales
rango de rotación entre distritos:  6.1 puntos porcentuales
El sector separa más que el distrito: coincide con hito7 (A. Solo sector ya da lift 1,78; añadir distrito sube poco).


## 8. Distribución de las variables del modelo y sus correlaciones

Las 9 variables del campeón `C. Sector + distrito + historia`
(`datos/ficha_modelo.json`). Se miran sobre las cohortes de modelado sin COVID.
Interesa: forma de cada distribución (colas, ceros masivos, censura) y si hay
variables redundantes entre sí.

In [10]:
num_vars = ["antiguedad_negocio", "antiguedad_local", "rotaciones_previas",
            "tasa_rotacion_local", "antiguedad_censurada"]
cat_vars = ["id_epigrafe", "id_division", "desc_distrito_local", "desc_tipo_acceso_local"]

display(d[num_vars].describe().T.round(3))

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, v in zip(axes.flat, num_vars):
    d[v].plot.hist(ax=ax, bins=30, color="#4C72B0")
    ax.set_title(v, fontsize=10); ax.set_ylabel("")
axes.flat[-1].axis("off")
fig.suptitle("Variables numéricas del campeón (cohortes de modelado, sin COVID)")
plt.show(); plt.close(fig)

corr = d[num_vars].corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_vars))); ax.set_xticklabels(num_vars, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(num_vars))); ax.set_yticklabels(num_vars, fontsize=8)
for i in range(len(num_vars)):
    for j in range(len(num_vars)):
        ax.text(j, i, f"{corr.values[i,j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Correlación entre variables numéricas"); fig.colorbar(im, ax=ax, shrink=.8)
plt.show(); plt.close(fig)

print("cardinalidad de las categóricas:")
for v in cat_vars:
    print(f"  {v:24s} {d[v].astype(str).str.strip().nunique():>5} valores distintos")

,count,mean,std,min,25%,50%,75%,max
antiguedad_negocio,479031.0,2.382,1.925,0.0,1.0,2.0,4.0,6.0
antiguedad_local,479031.0,2.649,1.969,0.0,1.0,2.0,4.0,6.0
rotaciones_previas,479031.0,0.099,0.336,0.0,0.0,0.0,0.0,5.0
tasa_rotacion_local,400618.0,0.036,0.126,0.0,0.0,0.0,0.0,1.0
antiguedad_censurada,479031.0,0.895,0.307,0.0,1.0,1.0,1.0,1.0


cardinalidad de las categóricas:
  id_epigrafe                453 valores distintos
  id_division                 91 valores distintos
  desc_distrito_local         22 valores distintos
  desc_tipo_acceso_local       3 valores distintos


**Lectura.** `antiguedad_negocio` y `antiguedad_local` están muy correlacionadas
(la mayoría de negocios no ha rotado, así que ambas edades coinciden) pero no
son idénticas. `rotaciones_previas` es cero para la gran mayoría —su señal está
en la cola—. `antiguedad_censurada` es una bandera: marca los locales cuya edad
real no se puede medir porque el negocio ya existía cuando arranca la serie
(2015). `id_epigrafe` tiene ~440 valores → alta cardinalidad, el motivo de usar
codificación ordinal con cajón para categorías raras.

## 9. Desbalanceo de la variable objetivo

La rotación es un evento **raro**: en torno al 4% al año. Cualquier métrica de
*accuracy* es inútil (un modelo que dice "nadie rota" acierta el 96%). Por eso
la métrica de decisión del proyecto es el **lift en el decil 10**
(`salida/RESULTADOS.md`).

In [11]:
g = ds.groupby("cohorte")["target"].agg(["size", "sum", "mean"])
g["ratio_neg_pos"] = ((g["size"] - g["sum"]) / g["sum"]).round(1)
g["% rotación"] = (g["mean"] * 100).round(2)
display(g[["size", "sum", "% rotación", "ratio_neg_pos"]])

sin_covid = ds[ds["uso"] != "covid"]
base = sin_covid["target"].mean()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
(g["mean"] * 100).plot.bar(ax=axes[0], color="#C44E52")
axes[0].axhline(base * 100, color="k", ls="--", lw=1, label=f"media sin COVID {base*100:.2f}%")
axes[0].set_title("Tasa de rotación por cohorte"); axes[0].set_ylabel("%"); axes[0].legend()
axes[0].tick_params(axis="x", rotation=0)
vc = sin_covid["target"].value_counts()
axes[1].bar(["se mantiene (0)", "rota (1)"], vc.values, color=["#55A868", "#C44E52"])
for i, v in enumerate(vc.values):
    axes[1].text(i, v, f"{v:,}\n{v/vc.sum()*100:.1f}%", ha="center", va="bottom", fontsize=9)
axes[1].set_title("Balance de clases (cohortes de modelado, sin COVID)")
axes[1].set_ylabel("nº de locales")
plt.show(); plt.close(fig)

print(f"desbalanceo global (sin COVID): 1 rotación por cada "
      f"{(sin_covid['target']==0).sum()/(sin_covid['target']==1).sum():.0f} locales que se mantienen")
covid = ds[ds['cohorte']==2020]['target'].mean()
print(f"cohorte COVID (2020->2021): {covid:.1%} — {covid/base:.1f}x la tasa normal, "
      "por eso queda fuera del modelado")

,size,sum,% rotación,ratio_neg_pos
cohorte,,,,
2015,78413,3478,4.44,21.5
2016,79049,3555,4.50,21.2
2017,79324,3689,4.65,20.5
2018,80187,3207,4.00,24.0
2019,80660,2851,3.53,27.3
2020,81122,13816,17.03,4.9
2021,81398,3234,3.97,24.2


desbalanceo global (sin COVID): 1 rotación por cada 23 locales que se mantienen
cohorte COVID (2020->2021): 17.0% — 4.1x la tasa normal, por eso queda fuera del modelado


---
## Síntesis para la memoria

| # | hallazgo | dónde se usa |
|---|---|---|
| 1 | Censo estable y creciente; escalón de cierres en 2020→2021 | R2, validación prospectiva |
| 2 | Comercio concentrado: 5 distritos y 2 secciones ≈ 2/3 del total | justificación del detalle sectorial |
| 3 | Epígrafe = cola muy larga (~440 valores, pocos con peso) | codificador con cajón `resto` (DATOS §6) |
| 4 | Reparto de situaciones estable año a año | viabilidad del entrenamiento con cohortes antiguas |
| 5 | 2014 sin rótulo fiable; OCR 0/O pre-2022; sección censal desde 2022 | arranque en 2015, límites de la fuente (DATOS §2) |
| 6 | Estructura espacial radial; ~X% de abiertos sin coordenada usable | variables de vecindario y su cobertura |
| 7 | El sector separa la rotación más que el distrito | orden de candidatos en R1 (A→B→C) |
| 8 | `antiguedad_negocio`≈`antiguedad_local`; rotaciones = evento de cola | selección y lectura de variables de historia |
| 9 | Target ~4%, desbalanceo ≈ 1:23; COVID (2020→2021) ~17%, ~4× | métrica = lift@10, exclusión de la cohorte COVID |

Las cuatro figuras para la memoria: `salida/figuras/eda_volumen_censo.png`,
`eda_situacion_evolucion.png`, `eda_densidad_espacial.png`,
`eda_rotacion_sector_distrito.png`.